In [ ]:
import pandas as pd

## Parte 1: Exploração inicial dos dados

In [ ]:
# carregando dados
df = pd.read_csv("../data/DOHMH_New_York_City_Restaurant_Inspection_Results.csv")
df.head() # por padrão só exibe as 5 primeiras linhas 

In [ ]:
print(df.shape)  # exibe o número de linhas e colunas

In [ ]:
## 2. tipos dos atributos;
df.dtypes 

In [ ]:
##  3. porcentagem de valores nulos por coluna;
nulos = (df.isnull().sum()/len(df))*100
nulos.sort_values(ascending=False)

In [ ]:
## 4. número de valores distintos por coluna;
df.nunique() #valores dististos 

In [ ]:
## 5. exemplos de valores frequentes para colunas categóricas;
categoricas = df.select_dtypes(include='object').columns

for coluna in categoricas: #a cada coluna aparece os 5 valores que mais apareceram
    print(f"\n{coluna}")
    print(df[coluna].value_counts().head(5)) 

In [ ]:
##  6. estatísticas simples para colunas numéricas e temporais.
df['INSPECTION DATE'] = pd.to_datetime(df['INSPECTION DATE'], errors='coerce')

print("Data mínima:", df['INSPECTION DATE'].min())
print("Data máxima:", df['INSPECTION DATE'].max())

## Parte 2: Escrita manual de regras candidatas

regras para identificar restrições: FD x identifica y; CFD depende de um caso pafra existir por exemplo algo que tem data de fechamento quando tem status fechado precisa de uma data de fechamento != NULL; DC ccasos que nunca deveriam aconetecer por exemplo uma data de inspeção ser registrada como um dia maior que o atual.


FD1: Cada restaurante identificado por um CAMIS possui um único nome comercial (DBA). ```CAMIS → DBA```

FD2: Cada CAMIS deve ter apenas 1 telefone. ```CAMIS → PHONE```

CFD1: GRADE 'A' precisa ter SCORE abaixo de 13. ```GRADE = 'A' ⇒ SCORE ≤ 13```

CFD2: BORO 'Manhattan' ZIPCODE comeca com 10. ```BORO = 'Manhattan' ⇒ ZIPCODE LIKE '10%'```

DC1: Nenhum restaurante pode ter SCORE negativo. ```¬(SCORE < 0)```

DC2: Não pode existir uma inspeção com data de inspeção posterior à data de registro.```¬(INSPECTION DATE > RECORD DATE)```

### Restrições de dados usando SQL

In [ ]:
import duckdb

duckdb.register("restaurants", df)

In [ ]:
duckdb.sql( """SELECT COUNT(*)
FROM (
    SELECT CAMIS
    FROM restaurants
    GROUP BY CAMIS
    HAVING COUNT(DISTINCT DBA) > 1
);""")



In [ ]:
duckdb.sql( """SELECT COUNT(*)
FROM (
    SELECT CAMIS
    FROM restaurants
    GROUP BY CAMIS
    HAVING COUNT(DISTINCT PHONE) > 1
);""")

In [ ]:
## A score of less than 14 points on either initial or re-inspection results in an “A”
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE GRADE = 'A'
  AND SCORE >= 14;""")

In [ ]:
## score of 14-27 points means a restaurant receives both a “B”
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE GRADE = 'B'
 AND (SCORE < 14 OR SCORE > 27);;""")

In [ ]:
## score of 28 or more points means a restaurant receives both a “C”
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE GRADE = 'C'
  AND SCORE < 28;""")

In [ ]:
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE BORO = 'Manhattan'
  AND CAST(ZIPCODE AS VARCHAR) NOT LIKE '10%';""")

In [ ]:
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE SCORE < 0; """)

In [ ]:
df["INSPECTION DATE"] = pd.to_datetime(df["INSPECTION DATE"])
df["RECORD DATE"] = pd.to_datetime(df["RECORD DATE"]) ##convertendo as datas

In [ ]:
duckdb.register("restaurants", df)

In [ ]:
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE "INSPECTION DATE" > "RECORD DATE"; """)